# ReLoG - Ablation 2: recensioni generate da LLM locale (Gemma 4 / Llama)

Ablation che simula lo scenario **senza recensioni reali**, ma dove ogni client
(dispositivo edge) genera *in locale* una recensione fittizia con un piccolo LLM
open-weight (Gemma 4 E2B via Ollama, oppure Llama). L'LLM riceve **stelle +
metadati item** e produce una recensione realistica; il suo embedding SBERT
alimenta la user tower al posto di quello reale.

Punti chiave (metodologici):
- **Generazione una volta sola e congelata**: le recensioni sono generate in un
  passo di preprocessing, salvate su disco (`cache`) e riusate identiche su tutti
  i seed. Cosi' la varianza tra seed resta confrontabile con la baseline e non
  viene inquinata dalla stocasticita' del LLM.
- **Determinismo**: `temperature` bassa + `seed` fisso; cache per coppia
  `(item, stelle)`, cioe' esattamente l'informazione disponibile in questo scenario.
- **Fallback**: se una chiamata al LLM fallisce, si ricade sul template
  deterministico dell'Ablation 1.
- **Generazione concorrente**: il server Ollama viene interrogato con piu'
  richieste in volo (`ThreadPoolExecutor`), per sfruttare gli slot paralleli
  della GPU (`OLLAMA_NUM_PARALLEL`, vedi setup sotto).

Rispetto al modello originale cambia **solo** la generazione del testo: item
tower, MLP locale, loop federato e valutazione restano identici.

In [1]:
# IMPORT

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import copy
import random
import math

In [2]:
# =============================================================================
# SEED
# =============================================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    # [OPT-9] benchmark=True accelera operazioni su tensori di dimensione fissa
    # (le torri hanno sempre input_dim=384, quindi CUDA può ottimizzare i kernel)
    torch.backends.cudnn.benchmark = True


In [38]:
# =============================================================================
# DATI
# =============================================================================

device = 'cuda' if torch.cuda.is_available() else 'cpu'

df_sampled      = pd.read_parquet('../../preprocessing/Movies_and_TV_review.parquet')
df_meta_aligned = pd.read_parquet('../../preprocessing/Movies_and_TV_meta.parquet')

NUM_USERS    = df_sampled['user_id_int'].nunique()
NUM_ITEMS    = df_sampled['item_id_int'].nunique()
INTERACTIONS = len(df_sampled)
SPARSITY     = (1 - (INTERACTIONS / (NUM_USERS * NUM_ITEMS))) * 100

print(f"Utenti: {NUM_USERS}, Item: {NUM_ITEMS}, Interactions: {INTERACTIONS}")
print(f"Sparsity: {SPARSITY:.2f}%")

Utenti: 4376, Item: 19199, Interactions: 50600
Sparsity: 99.94%


## Generazione recensioni (LLM locale) + Embedding

### Setup una tantum sul server (terminale, non nel notebook)

```bash
# 1) installa Ollama (se non gia' presente)
curl -fsSL https://ollama.com/install.sh | sh

# 2) scarica il modello edge (~1.8GB quantizzato)
ollama pull gemma4:e2b

# 3) verifica se ollama gira come servizio systemd (comune su Lightning/immagini preconfigurate)
systemctl status ollama --no-pager

# 4a) SE e' un servizio systemd: abilita il parallelismo in modo persistente
sudo systemctl edit ollama
#     -> nell'editor incolla:
#        [Service]
#        Environment="OLLAMA_NUM_PARALLEL=8"
sudo systemctl daemon-reload
sudo systemctl restart ollama

# 4b) SE NON e' un servizio systemd, avvialo a mano:
# OLLAMA_NUM_PARALLEL=8 nohup ollama serve > /tmp/ollama.log 2>&1 &

# 5) verifica che il parallelismo sia attivo: dopo una richiesta di prova,
#    il processo llama-server deve mostrare "-np 8" (non "-np 1")
ollama run gemma4:e2b "ciao"
ps aux | grep llama-server
```

`N_WORKERS` nel notebook deve combaciare (o essere <=) con `OLLAMA_NUM_PARALLEL`
impostato lato server: se il server ha 8 slot ma il notebook ne usa 4, sprechi
meta' della capacita' disponibile; se il notebook ne usa piu' di quanti il
server ne offre, le richieste in eccesso si accodano (nessun errore, solo
nessun guadagno aggiuntivo).

In [33]:
# =============================================================================
# ABLATION 2 — GENERAZIONE RECENSIONI CON LLM LOCALE (Qwen 0.5B via Ollama)
# =============================================================================
# NB: la generazione e' un passo di PREPROCESSING (fatto una volta, poi in cache).

import os
import pandas as pd
from tqdm import tqdm

# ----------------------------- CONFIG ----------------------------------------
MODEL_NAME     = "qwen2.5:0.5b"
CACHE_PATH     = "synthetic_reviews_llm_cache.parquet"
GEN_TEMPERATURE = 0.3
GEN_SEED        = 42
MAX_NEW_TOKENS  = 130
CHECKPOINT_EVERY = 20            # [FIX] checkpoint frequente: minimizza perdita in caso di interruzione

N_WORKERS = 8

# [DEBUG] tienilo False per il run completo: True solo per sanity check
DEBUG_LLM_ERRORS = False
# -----------------------------------------------------------------------------

# --- Descrittore item per il PROMPT (ricco: l'LLM riassume lui) --------------
def get_prompt_descriptor(meta_text, max_words=50):
    if not isinstance(meta_text, str) or not meta_text.strip():
        return ""
    return " ".join(meta_text.split()[:max_words])

prompt_desc_map = dict(
    zip(df_meta_aligned['item_id_int'],
        df_meta_aligned['meta_text'].map(get_prompt_descriptor))
)

# --- Titolo item per il FALLBACK a template ----------------------------------
def get_title(meta_text, max_words=50):
    if not isinstance(meta_text, str) or not meta_text.strip():
        return ""
    title = meta_text.split('.')[0].strip()
    return " ".join(title.split()[:max_words])

title_map = dict(
    zip(df_meta_aligned['item_id_int'],
        df_meta_aligned['meta_text'].map(get_title))
)

# --- Template di fallback (identico all'Ablation 1) --------------------------
SENTIMENT_TEMPLATES = {
    5: ["I absolutely love this. It's excellent and exactly what I hoped for. Highly recommended.",
        "Fantastic product, I'm extremely satisfied. It works perfectly and I would buy it again."],
    4: ["A good product that I'm happy with. It works well, with only minor drawbacks.",
        "Solid and reliable, I like it. Not perfect, but it does what it promises."],
    3: ["This product is okay. It's average: it does the job but nothing special.",
        "Mixed feelings about this one. It's decent but it has a few shortcomings."],
    2: ["I'm disappointed with this product. It has problems and didn't meet my expectations.",
        "Not great. It works poorly and I expected more from it."],
    1: ["This product is terrible. It doesn't work as expected and I would not recommend it.",
        "Very poor quality, I regret buying it. It failed to do what it should."],
}

def template_fallback(stars, item_id):
    variants = SENTIMENT_TEMPLATES[stars]
    sentiment = variants[int(item_id) % len(variants)]
    return f"{sentiment} {title_map.get(int(item_id), '')}".strip()

# --- Prompt per l'LLM --------------------------------------------------------
SYSTEM_PROMPT = (
    "You are a customer writing a product review for an e-commerce website. "
    "Given a product and the star rating you gave it, write a short, realistic "
    "review of 2-3 sentences that is clearly consistent with that rating "
    "(enthusiastic for 5 stars, negative for 1 star, mixed for 3). "
    "Write your OWN opinion in your OWN words: do NOT copy or repeat the product "
    "description, specifications, or technical details verbatim. "
    "STRICT LIMIT: no more than 30 words total, never exceed it. "
    "Output ONLY the review text: no preamble, no quotes, no rating number, no lists, "
    "no reasoning, no explanation of your choice."
)

def build_prompt(stars, descriptor):
    return (f"Product: {descriptor}\n"
            f"My rating: {stars} out of 5 stars.\n\n"
            f"Write my review:")

# --- Pulizia output -----------------------------------------------------------
_CONTROL_TOKENS = [
    "<|im_end|>", "<|im_start|>", "<end_of_turn>", "<start_of_turn>",
    "<|end_of_turn|>", "<|start_of_turn|>", "<|think|>", "<|/think|>",
    "<|eot_id|>", "<|start_header_id|>", "<|end_header_id|>", "<|begin_of_text|>",
    "<br>", "<br/>", "<br />",
]

def clean_review(text):
    if not text:
        return ""
    t = text
    for tok in _CONTROL_TOKENS:
        t = t.replace(tok, "")
    lower = t.lower()
    marker = "done thinking."
    if marker in lower:
        idx = lower.index(marker) + len(marker)
        t = t[idx:]
    t = " ".join(t.strip().split())
    t = t.strip().strip('"').strip("'").strip()
    for p in ("review:", "here is a review:", "here's a review:",
              "sure,", "certainly,", "okay,"):
        if t.lower().startswith(p):
            t = t[len(p):].strip()
    words = t.split()
    if len(words) > 100:          # [FIX] tetto alzato: prima tagliava a 60 parole
                                # SEMPRE, indipendentemente da MAX_NEW_TOKENS,
                                # producendo frasi mozzate a meta'. 100 e' generoso
                                # ma resta una rete di sicurezza contro output degeneri.
        t = " ".join(words[:100])
    return t

FEWSHOT_EXAMPLES = [
    {"role": "user", "content": "Product: Wireless Bluetooth Earbuds, sweat resistant.\nMy rating: 5 out of 5 stars.\n\nWrite my review:"},
    {"role": "assistant", "content": "These earbuds completely exceeded my expectations! Comfortable fit, great sound, and they never fall out during workouts."},
    {"role": "user", "content": "Product: USB Charging Cable, braided nylon.\nMy rating: 2 out of 5 stars.\n\nWrite my review:"},
    {"role": "assistant", "content": "Disappointed with this cable. It stopped charging properly after two weeks of light use. Wouldn't recommend it."},
]

# --- Filtri di qualita' -------------------------------------------------------
# [FIX] item con metadati non informativi: vanno dritto al fallback,
# senza nemmeno chiamare l'LLM (non ha nulla su cui basare la recensione)
_PLACEHOLDER_MARKERS = ("description unavailable", "no description available",
                        "product description not available", "n/a")

# [FIX] rileva rifiuti/risposte meta del modello (non recensioni vere)
_REFUSAL_MARKERS = ("i'm sorry", "i cannot", "i can't assist",
                    "as an ai", "i am unable", "i don't have enough")

# --- Client Ollama -----------------------------------------------------------
import ollama

def _extract_content(resp):
    try:
        return resp["message"]["content"]
    except (TypeError, KeyError):
        return resp.message.content

# [FIX] rileva copia-incolla del testo prodotto invece di una recensione vera:
# se una porzione consistente dell'output combacia col descrittore originale,
# il modello ha "ricopiato" invece di scrivere.
def _looks_copied(review_text, descriptor, min_shared_words=8):
    review_words = review_text.lower().split()
    descriptor_words = descriptor.lower().split()
    # cerca la piu' lunga sequenza di parole consecutive condivisa
    max_run = 0
    desc_set_windows = set()
    for i in range(len(descriptor_words) - min_shared_words + 1):
        desc_set_windows.add(tuple(descriptor_words[i:i+min_shared_words]))
    for i in range(len(review_words) - min_shared_words + 1):
        if tuple(review_words[i:i+min_shared_words]) in desc_set_windows:
            return True
    return False

def generate_review_llm(stars, item_id):
    """Ritorna una tupla (review_text, source), dove source e' 'llm' o 'fallback'.
    Cosi' possiamo quantificare a fine run quante recensioni vengono davvero
    dall'LLM e quante dal template deterministico."""
    descriptor = prompt_desc_map.get(int(item_id), "")

    # [FIX] descrittore vuoto o placeholder non informativo -> fallback diretto
    if not descriptor.strip() or any(m in descriptor.lower() for m in _PLACEHOLDER_MARKERS):
        return template_fallback(stars, item_id), "fallback"

    try:
        resp = ollama.chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}]
                     + FEWSHOT_EXAMPLES
                     + [{"role": "user", "content": build_prompt(stars, descriptor)}],
            think=False,
            options={"temperature": GEN_TEMPERATURE,
                     "seed": GEN_SEED,
                     "num_predict": MAX_NEW_TOKENS},
        )

        done_reason = resp.get("done_reason")
        if done_reason == "length":
            raise ValueError(f"generazione troncata dal server (done_reason=length)")
            
        text = clean_review(_extract_content(resp))

        if len(text.split()) < 3:
            raise ValueError(f"output troppo corto: {text!r}")

        if any(m in text.lower() for m in _REFUSAL_MARKERS):
            raise ValueError(f"risposta meta/rifiuto rilevata: {text!r}")

        if _looks_copied(text, descriptor):
            raise ValueError(f"output copia-incolla del prodotto, non recensione: {text!r}")

        return text, "llm"
    except Exception as e:
        if DEBUG_LLM_ERRORS:
            import traceback
            print(f"\n[DEBUG] Fallback scattato per item={item_id}, stars={stars}")
            print(f"[DEBUG] Eccezione: {type(e).__name__}: {e}")
            traceback.print_exc()
        return template_fallback(stars, item_id), "fallback"

# --- Verifica connessione al server Ollama (fail-fast con messaggio chiaro) ---
try:
    _ = ollama.chat(model=MODEL_NAME,
                    messages=[{"role": "user", "content": "ping"}],
                    options={"num_predict": 1})
    print(f"Ollama OK — modello '{MODEL_NAME}' raggiungibile.")
except Exception as e:
    raise RuntimeError(
        f"Impossibile contattare Ollama con il modello '{MODEL_NAME}'.\n"
        f"Assicurati che il server sia attivo e che il modello sia scaricato "
        f"('ollama pull {MODEL_NAME}'). Errore: {e}"
    )

Ollama OK — modello 'qwen2.5:0.5b' raggiungibile.


In [34]:
# =============================================================================
# TEST RAPIDO — verifica manuale su pochi esempi PRIMA del run completo
# =============================================================================
# Genera (senza toccare la cache) qualche recensione di esempio, per controllare
# a occhio che non ci siano artefatti di chat-template o testo di "ragionamento"
# visibile prima di lanciare le decine di migliaia di chiamate del run completo.

import random as _random

_sample_pairs = df_sampled[['item_id_int', 'rating']].drop_duplicates().sample(
    n=min(8, len(df_sampled)), random_state=0
)

print(f"Test di sanity su {len(_sample_pairs)} coppie (item, stelle):\n")
for _, row in _sample_pairs.iterrows():
    stars = max(1, min(5, int(round(float(row['rating'])))))
    item_id = int(row['item_id_int'])
    review, source = generate_review_llm(stars, item_id)   # [FIX] unpack tupla
    print(f"[{stars}*] item {item_id} | source={source}")
    print(f"   -> {review}")
    print()

print("Controlla che NON compaiano: tag come <|im_end|>, blocchi 'Thinking...'/"
      "'done thinking.', frasi di preambolo (es. 'Sure, here is...').")
print("Se tutto ok, procedi con la cella successiva (generazione completa).")

Test di sanity su 8 coppie (item, stelle):

[5*] item 2018 | source=llm
   -> Absolutely loved this! The movies and TV content is great for kids and families, and the features are perfect for featured categories in our e-commerce website.

[4*] item 8506 | source=llm
   -> This is an excellent boxset! The characters are well-developed, the storyline is engaging, and the music is great. I highly recommend it to fans of the show.

[2*] item 1761 | source=llm
   -> This movie is not worth watching. It's a classic but outdated and too mature for me to watch. The running time was long, so I couldn't finish it in one sitting.

[3*] item 17315 | source=llm
   -> The movie was well-received but the action sequences were too fast for me. The plot was confusing, and the characters weren't developed enough. Overall, it's a decent film but not as good as some other movies in this genre.

[5*] item 9551 | source=llm
   -> Absolutely loved this movie! The story is incredible, the characters are rela

In [35]:
import random
_diag_sample = df_sampled[['item_id_int', 'rating']].drop_duplicates().sample(n=50, random_state=7)
n_fallback = 0
for _, row in _diag_sample.iterrows():
    stars = max(1, min(5, int(round(float(row['rating'])))))
    _, source = generate_review_llm(stars, int(row['item_id_int']))
    if source == "fallback":
        n_fallback += 1
print(f"Fallback su 50 campioni: {n_fallback}/50 ({100*n_fallback/50:.0f}%)")

Fallback su 50 campioni: 2/50 (4%)


In [39]:
import time
from concurrent.futures import ThreadPoolExecutor

def _timed_call(i):
    t0 = time.time()
    generate_review_llm(5, i % 100)
    return time.time() - t0

pairs = df_sampled[['item_id_int', 'rating']].copy()
pairs['rating_int'] = pairs['rating'].map(_stars)
unique_pairs = pairs[['item_id_int', 'rating_int']].drop_duplicates()
print(f"Coppie (item, stelle) uniche nel dataset: {len(unique_pairs)}")

t_start = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    durations = list(ex.map(_timed_call, range(8)))
t_total = time.time() - t_start
print(f"Tempo totale 8 richieste parallele: {t_total:.2f}s | media singola: {sum(durations)/len(durations):.2f}s")
print(f"Throughput: {8/t_total:.2f} it/s -> su {len(unique_pairs)} coppie: {len(unique_pairs)/(8/t_total)/3600:.1f} ore stimate")

Coppie (item, stelle) uniche nel dataset: 27922
Tempo totale 8 richieste parallele: 3.76s | media singola: 3.45s
Throughput: 2.13 it/s -> su 27922 coppie: 3.6 ore stimate


In [40]:
# =============================================================================
# GENERAZIONE COMPLETA (con cache + esecuzione concorrente + tracking source)
# =============================================================================

def _stars(r):
    return max(1, min(5, int(round(float(r)))))

# --- Cache: carica se esiste --------------------------------------------------
# [FIX] la cache ora contiene anche la colonna 'source'
if os.path.exists(CACHE_PATH):
    _cache_df = pd.read_parquet(CACHE_PATH)
    review_cache = {
        (int(r.item_id_int), int(r.rating_int)): (r.review, r.source)
        for r in _cache_df.itertuples(index=False)
    }
    print(f"Cache trovata: {len(review_cache)} coppie (item, stelle) gia' generate.")
else:
    review_cache = {}

pairs = df_sampled[['item_id_int', 'rating']].copy()
pairs['rating_int'] = pairs['rating'].map(_stars)
unique_pairs = pairs[['item_id_int', 'rating_int']].drop_duplicates()
print(f"Coppie (item, stelle) uniche nel dataset: {len(unique_pairs)}")

def _save_cache():
    df_out = pd.DataFrame(
        [{"item_id_int": k[0], "rating_int": k[1], "review": v[0], "source": v[1]}
         for k, v in review_cache.items()]
    )
    df_out.to_parquet(CACHE_PATH, index=False)

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

todo = [(int(i), int(s)) for i, s in unique_pairs.itertuples(index=False)
        if (int(i), int(s)) not in review_cache]
print(f"Da generare: {len(todo)} / {len(unique_pairs)} coppie "
      f"(gia' in cache: {len(unique_pairs) - len(todo)})")

new_count = 0
_save_lock = threading.Lock()

if todo:
    try:
        with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
            future_to_key = {
                executor.submit(generate_review_llm, stars, item_id): (item_id, stars)
                for item_id, stars in todo
            }
            for future in tqdm(as_completed(future_to_key), total=len(todo),
                               desc=f"LLM reviews [{MODEL_NAME}] x{N_WORKERS} workers"):
                item_id, stars = future_to_key[future]
                review_cache[(item_id, stars)] = future.result()   # (text, source)
                new_count += 1
                if new_count % CHECKPOINT_EVERY == 0:
                    with _save_lock:
                        _save_cache()
    finally:
        # [FIX] garantisce il salvataggio ANCHE se interrompi il kernel a mano
        if new_count > 0:
            with _save_lock:
                _save_cache()
            print(f"[CHECKPOINT] Salvate {new_count} nuove recensioni prima di uscire.")

if new_count > 0:
    print(f"Generate {new_count} nuove recensioni. Cache aggiornata: {CACHE_PATH}")
else:
    print("Nessuna nuova recensione da generare (tutto in cache).")

# --- Mappa recensione + source su ogni interazione ---------------------------
df_sampled['synthetic_review'] = [
    review_cache[(int(i), _stars(r))][0]
    for i, r in zip(df_sampled['item_id_int'], df_sampled['rating'])
]
df_sampled['review_source'] = [
    review_cache[(int(i), _stars(r))][1]
    for i, r in zip(df_sampled['item_id_int'], df_sampled['rating'])
]

# [FIX] riepilogo finale: quante recensioni vengono davvero dall'LLM
n_llm = (df_sampled['review_source'] == 'llm').sum()
n_fallback = (df_sampled['review_source'] == 'fallback').sum()
print(f"\nRiepilogo fonte recensioni: LLM={n_llm} ({100*n_llm/len(df_sampled):.1f}%), "
      f"fallback={n_fallback} ({100*n_fallback/len(df_sampled):.1f}%)")

print("Esempi di recensioni generate, per livello di stelle:\n")

for stars in [5, 2, 3]:
    subset = df_sampled[df_sampled['rating'].round().astype(int) == stars]
    print(f"--- {stars} stelle ({len(subset)} interazioni totali) ---")
    for _, row in subset.head(3).iterrows():
        print(f"  [source={row['review_source']}] {row['synthetic_review']}")
    print()

Coppie (item, stelle) uniche nel dataset: 27922
Da generare: 27922 / 27922 coppie (gia' in cache: 0)


LLM reviews [qwen2.5:0.5b] x8 workers: 100%|██████████| 27922/27922 [2:36:30<00:00,  2.97it/s]  


[CHECKPOINT] Salvate 27922 nuove recensioni prima di uscire.
Generate 27922 nuove recensioni. Cache aggiornata: synthetic_reviews_llm_cache.parquet

Riepilogo fonte recensioni: LLM=49281 (97.4%), fallback=1319 (2.6%)
Esempi di recensioni generate, per livello di stelle:

--- 5 stelle (29970 interazioni totali) ---
  [source=llm] Absolutely loved this set! The Star Trek Enterprise - The Complete First Season is a must-have for anyone interested in Star Trek. It's packed with episodes and the quality is top-notch.
  [source=llm] Absolutely loved this book! The characters were well-developed and the plot kept me hooked until the very end. Highly recommend for fans of science fiction and history.
  [source=llm] Absolutely loved this movie! It was a great way to unwind after work. Would definitely recommend it to others.

--- 2 stelle (2546 interazioni totali) ---
  [source=llm] These movies are great, but the quality is not up to par. I'd be disappointed if they were for me.
  [source=llm]

In [20]:
import time

def bench_model_isolated(model, n_calls=4):
    """Chiama lo STESSO modello n_calls volte di fila, senza mai cambiarlo:
    isola la velocita' di generazione sostenuta dal costo di caricamento/swap."""
    test_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(5, "Test item, a sample product for benchmarking.")},
    ]
    print(f"\n=== {model} ===")
    for i in range(n_calls):
        resp = ollama.chat(model=model, messages=test_messages, think=False,
                           options={"temperature": 0.3, "seed": 42, "num_predict": 90})
        eval_count  = resp.get("eval_count")
        eval_dur_ns = resp.get("eval_duration")
        load_dur_s  = (resp.get("load_duration") or 0) / 1e9
        tok_per_sec = eval_count / (eval_dur_ns/1e9) if (eval_count and eval_dur_ns) else 0
        tag = "1a chiamata (puo' includere caricamento)" if i == 0 else f"chiamata {i+1} (a caldo)"
        print(f"  {tag:42s} | tok/s={tok_per_sec:6.1f} | load={load_dur_s:.2f}s")


In [41]:
# =============================================================================
# EMBEDDINGS SBERT — sulle recensioni generate dall'LLM (Ablation 2)
# =============================================================================
# Identico alla baseline, ma la user tower riceve l'embedding della recensione
# SINTETICA (df_sampled['synthetic_review']) invece di quella reale.
# L'item tower resta INVARIATA (usa sempre meta_text completo).

sbert = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print("Calcolo embeddings recensioni sintetiche (LLM)...")
review_embeddings = sbert.encode(
    df_sampled['synthetic_review'].tolist(),
    batch_size=64, show_progress_bar=True, convert_to_numpy=True
)
review_emb_map = {i: review_embeddings[i] for i in range(len(df_sampled))}

print("Calcolo embeddings metadati item...")
meta_embeddings = sbert.encode(
    df_meta_aligned['meta_text'].tolist(),
    batch_size=64, show_progress_bar=True, convert_to_numpy=True
)
item_meta_tensor = torch.tensor(meta_embeddings, dtype=torch.float32)
print(f"Item bank: {item_meta_tensor.shape}")

# [OPT-1] Pre-carica item_meta_tensor su GPU UNA VOLTA SOLA qui.
item_meta_tensor_gpu = item_meta_tensor.to(device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Calcolo embeddings recensioni sintetiche (LLM)...


Batches:   0%|          | 0/791 [00:00<?, ?it/s]

Calcolo embeddings metadati item...


Batches:   0%|          | 0/300 [00:00<?, ?it/s]

Item bank: torch.Size([19199, 384])


##  Definizione architettura

In [42]:
# =============================================================================
# ARCHITETTURA
# =============================================================================

class UserTower(nn.Module):
    def __init__(self, input_dim=384, hidden_dim=512, output_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            # nn.Dropout(0.1),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


class ItemTower(nn.Module):
    def __init__(self, input_dim=384, hidden_dim=512, output_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            # nn.Dropout(0.1),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, text_emb):
        return F.normalize(self.net(text_emb), dim=-1)


class LocalScoreFunction(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, user_emb, item_emb):
        if user_emb.shape[0] != item_emb.shape[0]:
            if user_emb.shape[0] == 1:
                user_emb = user_emb.expand(item_emb.shape[0], -1)
            else:
                raise RuntimeError(f"Shape mismatch: {user_emb.shape} vs {item_emb.shape}")
        x = torch.cat([user_emb, item_emb], dim=-1)
        return self.net(x)


class TwoTowerRecommender(nn.Module):
    def __init__(self, input_dim=384, hidden_dim=512, output_dim=256, inference_temperature=0.07):
        super().__init__()
        self.item_tower = ItemTower(input_dim, hidden_dim, output_dim)
        self.user_tower = UserTower(input_dim, hidden_dim, output_dim)
        self.client_mlp = LocalScoreFunction(input_dim=output_dim * 2, hidden_dim=128)
        self.inference_temperature = inference_temperature

    def get_user_repr(self, review_embeddings):
        return self.user_tower(review_embeddings).mean(dim=0, keepdim=True)

    def get_item_repr(self, item_meta_embeddings):
        return self.item_tower(item_meta_embeddings)

    def training_score(self, user_repr, item_reprs):
        raw_scores = self.client_mlp(user_repr, item_reprs).squeeze(-1)
        return raw_scores / self.inference_temperature

    def score(self, user_repr, item_reprs):
        return self.training_score(user_repr, item_reprs)


def bpr_loss(pos_scores, neg_scores):
    # [OPT-5] BPR loss originale usava repeat_interleave + broadcasting implicito
    # che creava tensori intermedi grandi inutilmente.
    # Questa versione usa unsqueeze+broadcasting diretto: più compatta e più veloce.
    # pos_scores: [N_pos], neg_scores: [N_neg]
    # diff: [N_pos, N_neg] via broadcasting — nessuna allocazione extra
    diff = pos_scores.unsqueeze(1) - neg_scores.unsqueeze(0)
    return -F.logsigmoid(diff).mean()

# Utility dati

In [43]:
def get_client_data(user_id, df, emb_map, mode="train"):
    user_df = df[df['user_id_int'] == user_id].sort_values('timestamp')
    if len(user_df) < 3:
        return None, None
    indices = user_df.index.tolist()
    train_idx = indices[:-2]
    val_idx   = indices[-2]
    test_idx  = indices[-1]
    X_train = torch.tensor(
        np.array([emb_map[i] for i in train_idx]),
        dtype=torch.float32
    )
    train_item_ids = user_df.loc[train_idx, 'item_id_int'].tolist()
    if mode == "val":
        target_id = int(user_df.loc[val_idx, 'item_id_int'])
    elif mode == "test":
        target_id = int(user_df.loc[test_idx, 'item_id_int'])
    else:
        target_id = None
    return (X_train, train_item_ids), target_id

In [44]:
# =============================================================================
# HARD NEGATIVE SAMPLING — OTTIMIZZATO
# =============================================================================

def sample_hard_negatives(user_repr, pos_set, all_metas, local_model,
                          num_neg, num_candidates, device, num_total_items):
    # [OPT-3] Il loop originale campionava con random.randint() uno alla volta
    # dentro un while, con controllo Python ad ogni iterazione (lento).
    # np.random.choice con replace=False campiona tutti i candidati in un colpo
    # solo, completamente in C, poi filtra con una maschera booleana vettorizzata.
    # Su num_candidates=500 questo è ~10-20x più veloce del loop Python.
    all_ids = np.arange(num_total_items)
    pos_arr = np.array(list(pos_set), dtype=np.int64)
    mask = np.ones(num_total_items, dtype=bool)
    mask[pos_arr] = False
    eligible = all_ids[mask]

    if len(eligible) < num_neg:
        return eligible.tolist()

    n_cands = min(num_candidates, len(eligible))
    candidate_ids = np.random.choice(eligible, size=n_cands, replace=False)

    with torch.no_grad():
        cand_ids_tensor = torch.tensor(candidate_ids, device=device)
        cand_metas = all_metas[cand_ids_tensor]
        cand_reprs = local_model.get_item_repr(cand_metas)
        cand_scores = local_model.training_score(user_repr.detach(), cand_reprs)

    top_k = min(num_neg, len(candidate_ids))
    top_indices = torch.topk(cand_scores, top_k).indices.cpu().numpy()
    return candidate_ids[top_indices].tolist()

## Train client

In [45]:
def get_lr(base_lr, current_step, warmup_steps, total_steps):
    """LR warmup lineare → cosine annealing."""
    if current_step < warmup_steps:
        return base_lr * (current_step + 1) / warmup_steps
    progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

def train_client(user_id, global_state_dict, X_train_reviews, train_item_ids,
                 all_metas_gpu,   # [OPT-1] riceve direttamente il tensore già su GPU
                 device, client_states,
                 lr=0.001, epochs=5, num_neg=10, use_hard_negatives=True,
                 lr_warmup_steps=10, current_step=0, total_steps=100):

    local_model = TwoTowerRecommender().to(device)
    local_model.load_state_dict(global_state_dict, strict=False)

    user_local_data = client_states.get(user_id, None)
    if user_local_data is not None:
        local_model.client_mlp.load_state_dict(user_local_data)

    effective_lr = get_lr(lr, current_step, lr_warmup_steps, total_steps)
    optimizer = torch.optim.Adam(local_model.parameters(), lr=effective_lr)
    local_model.train()

    # [OPT-1] all_metas è già su GPU — nessun trasferimento qui
    X_train = X_train_reviews.to(device)
    pos_set = set(train_item_ids)
    num_total_items = all_metas_gpu.shape[0]

    # [OPT-4] pos_tensor costruito UNA VOLTA fuori dal loop delle epoche.
    # Nel codice originale veniva ricostruito a ogni epoca (allocazione GPU inutile).
    pos_tensor = torch.tensor(train_item_ids, device=device)
    pos_metas  = all_metas_gpu[pos_tensor]   # shape: [N_pos, 384] — fisso per tutte le epoche

    loss = None
    for _ in range(epochs):
        optimizer.zero_grad()
        user_repr = local_model.get_user_repr(X_train)

        # pos_metas è già pronto — solo forward pass
        pos_reprs = local_model.get_item_repr(pos_metas)

        if use_hard_negatives:
            neg_ids = sample_hard_negatives(
                user_repr, pos_set, all_metas_gpu, local_model,
                num_neg=len(train_item_ids) * num_neg,
                num_candidates=2000, device=device,
                num_total_items=num_total_items
            )
        else:
            # [OPT-3] anche il campionamento random semplice vettorizzato
            all_ids = np.arange(num_total_items)
            mask = np.ones(num_total_items, dtype=bool)
            mask[np.array(list(pos_set))] = False
            neg_ids = np.random.choice(
                all_ids[mask],
                size=len(train_item_ids) * num_neg,
                replace=True
            ).tolist()

        neg_tensor = torch.tensor(neg_ids, device=device)
        neg_metas  = all_metas_gpu[neg_tensor]
        neg_reprs  = local_model.get_item_repr(neg_metas)

        pos_scores = local_model.training_score(user_repr, pos_reprs)
        neg_scores = local_model.training_score(user_repr, neg_reprs)

        # [OPT-5] BPR loss con broadcasting diretto (vedi definizione sopra)
        loss = bpr_loss(pos_scores, neg_scores)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(local_model.parameters(), 1.0)
        optimizer.step()

    # [OPT-10] Salviamo lo state_dict della client_mlp direttamente (no deepcopy
    # di tutto il modello). deepcopy è lento perché copia ricorsivamente tutti
    # gli attributi Python; state_dict() è già una copia piatta dei tensori.
    client_states[user_id] = local_model.client_mlp.state_dict()

    shared_state = {k: v.cpu() for k, v in local_model.state_dict().items()
                    if 'client_mlp' not in k}

    return shared_state, loss.item(), len(train_item_ids)

In [46]:
# =============================================================================
# FEDAVG CON MOMENTUM — OTTIMIZZATO
# =============================================================================

def weighted_fedavg_momentum(global_model, local_weights_list, local_sizes,
                             momentum_buffer, beta=0.9):
    # [OPT-6] Il codice originale faceva deepcopy dell'intero state_dict globale
    # come punto di partenza, poi riallocava layer_avg da zero per ogni chiave.
    # Questa versione lavora direttamente sullo state_dict corrente in-place
    # con operazioni torch, evitando allocazioni Python extra.
    total_samples = sum(local_sizes)
    global_dict   = global_model.state_dict()
    keys_to_agg   = [k for k in global_dict if 'client_mlp' not in k]

    if momentum_buffer is None:
        momentum_buffer = {k: torch.zeros_like(global_dict[k]) for k in keys_to_agg}

    with torch.no_grad():
        for key in keys_to_agg:
            # Media pesata in-place: accumula direttamente su un tensore zero
            layer_avg = torch.zeros_like(global_dict[key])
            for i, w in enumerate(local_weights_list):
                # i pesi locali sono già su CPU (vedi train_client)
                layer_avg.add_(w[key].to(layer_avg.device),
                               alpha=local_sizes[i] / total_samples)

            delta = layer_avg - global_dict[key]
            momentum_buffer[key].mul_(beta).add_(delta, alpha=1 - beta)
            global_dict[key].add_(momentum_buffer[key])

    # Ricarica in-place senza ricostruire il modello
    global_model.load_state_dict(global_dict, strict=False)
    return momentum_buffer

## Validation

In [47]:
# =============================================================================
# VALUTAZIONE — OTTIMIZZATA
# =============================================================================

def evaluate_top_k(global_model, eval_users, df, emb_map,
                   all_metas_gpu,   # [OPT-1] tensore già su GPU
                   client_states, k=10, device='cuda', mode="test",
                   eval_fraction=1.0):

    if eval_fraction < 1.0:
        n_sample   = max(1, int(len(eval_users) * eval_fraction))
        eval_users = np.random.choice(eval_users, n_sample, replace=False)
    
    num_items    = all_metas_gpu.shape[0]
    global_state = global_model.state_dict()
    all_ids_set  = set(range(num_items))
    all_ids_arr  = np.arange(num_items)

    # [OPT-7] Pre-computa gli embedding di TUTTI gli item UNA VOLTA SOLA.
    # Nel codice originale ogni utente ricalcolava get_item_repr su tutto
    # l'item bank durante l'inferenza (N_users * N_items forward pass sulla
    # ItemTower). Con la pre-computazione, la ItemTower viene eseguita solo
    # una volta e il risultato viene riusato per ogni utente: risparmio enorme
    # specialmente con item bank grandi (es. 50k item, 500 utenti eval =
    # 25M forward pass → 50k forward pass).
    global_model.eval()
    with torch.no_grad():
        # Processa in chunk per non saturare la VRAM
        chunk = 4096
        all_item_embs = torch.cat([
            global_model.get_item_repr(all_metas_gpu[i:i + chunk])
            for i in range(0, num_items, chunk)
        ], dim=0)  # [N_items, output_dim]

    hits, ndcgs, count = 0, 0, 0

    use_amp = (device == 'cuda')

    for user_id in tqdm(eval_users, desc=f"Evaluating ({mode})"):
        train_data, target_id = get_client_data(user_id, df, emb_map, mode=mode)
        if train_data is None:
            continue
        X_train, train_ids = train_data

        local_model = TwoTowerRecommender().to(device)
        local_model.load_state_dict(global_state, strict=False)

        user_local_data = client_states.get(user_id, None)
        if user_local_data is not None:
            local_model.client_mlp.load_state_dict(user_local_data)

        lr_eval = 0.005

        # --- LOCAL FINETUNING ---
        local_model.train()
        optimizer = torch.optim.Adam(local_model.client_mlp.parameters(), lr=lr_eval)
        X_train_dev = X_train.to(device)

        # [OPT-4] pos_tensor fuori dal loop epoche
        batch_pos = train_ids if len(train_ids) < 32 else random.sample(train_ids, 32)
        pos_t = torch.tensor(batch_pos, device=device)

        train_ids_set = set(train_ids)
        eligible_neg  = all_ids_arr[~np.isin(all_ids_arr, list(train_ids_set))]

        # [OPT-8] autocast fp16 per il finetuning locale (3 epoche leggere)
        # Riduce uso memoria e accelera su GPU con Tensor Cores (Ampere/Turing+)
        with torch.amp.autocast(device_type=device, enabled=use_amp):
            for _ in range(3):
                optimizer.zero_grad()
                user_repr = local_model.get_user_repr(X_train_dev)
                pos_reprs = local_model.get_item_repr(all_metas_gpu[pos_t])

                # [OPT-3] campionamento negativo vettorizzato
                neg_idx = np.random.choice(eligible_neg, size=len(batch_pos), replace=False)
                neg_t   = torch.tensor(neg_idx, device=device)
                neg_reprs = local_model.get_item_repr(all_metas_gpu[neg_t])

                loss = bpr_loss(
                    local_model.training_score(user_repr, pos_reprs),
                    local_model.training_score(user_repr, neg_reprs)
                )
                loss.backward()
                optimizer.step()

        # --- INFERENZA ---
        local_model.eval()
        with torch.no_grad():
            user_repr = local_model.get_user_repr(X_train_dev)

            # [OPT-7] Usa gli embedding pre-calcolati invece di rieseguire la ItemTower.
            # Applichiamo solo il client_mlp (che è specifico dell'utente e non può
            # essere pre-calcolato) sugli embedding già disponibili.
            # Questo è il risparmio più grande in evaluate_top_k.
            neg_cands   = list(all_ids_set - train_ids_set - {target_id})
            neg_arr     = np.array(neg_cands)
            neg_embs    = all_item_embs[neg_arr]  # shape: [N_neg, output_dim]
            target_emb  = all_item_embs[target_id].unsqueeze(0)  # [1, output_dim]

            # Score target e negativi in un unico batch
            # [OPT-8] autocast per l'inferenza finale
            with torch.amp.autocast(device_type=device, enabled=use_amp):
                target_score = local_model.score(user_repr, target_emb).item()
                neg_scores   = local_model.score(user_repr, neg_embs)

            all_scores = torch.cat([
                torch.tensor([target_score], device=device),
                neg_scores
            ])
            top_k_idx = torch.topk(all_scores, k).indices.cpu().numpy()

            if 0 in top_k_idx:
                hits += 1
                rank = int(np.where(top_k_idx == 0)[0][0])
                ndcgs += 1.0 / np.log2(rank + 2)
            count += 1

    if count == 0:
        return 0.0, 0.0
    return hits / count, ndcgs / count

In [48]:
# =============================================================================
# VALUTAZIONE FEW-SHOT (unseen users)
# =============================================================================
# Protocollo:
#   - Le prime `num_shots` interazioni temporali dell'utente sono usate
#     per il fine-tuning della client_mlp (partendo dai pesi globali).
#   - Il target è la (num_shots+1)-esima interazione.
#   - Gli utenti con meno di (num_shots+2) interazioni vengono saltati
#     (servono almeno num_shots per il finetune + 1 target + 1 di margine).
#
# num_shots=None → "full": tutte le interazioni tranne l'ultima per finetune,
#                           l'ultima come target (stesso protocollo dei warm users).
 
def get_fewshot_data(user_id, df, emb_map, num_shots):
    """
    Ritorna (X_shots, shot_item_ids, target_id) per un unseen user.
    X_shots: embeddings delle prime num_shots review [num_shots, 384]
    shot_item_ids: item ids corrispondenti
    target_id: item id della (num_shots+1)-esima interazione
    """
    user_df = df[df['user_id_int'] == user_id].sort_values('timestamp')
 
    if num_shots is None:
        # Full: tutto tranne l'ultima
        if len(user_df) < 2:
            return None, None, None
        indices       = user_df.index.tolist()
        shot_idx      = indices[:-2] # per comparazione con warm users
        target_idx    = indices[-1]
    else:
        # K-shot: prime K + target alla posizione K+1
        min_required = num_shots + 1
        if len(user_df) < min_required:
            return None, None, None
        indices    = user_df.index.tolist()
        shot_idx   = indices[:num_shots]
        target_idx = indices[num_shots]  # la (K+1)-esima
 
    X_shots = torch.tensor(
        np.array([emb_map[i] for i in shot_idx]),
        dtype=torch.float32
    )
    shot_item_ids = user_df.loc[shot_idx, 'item_id_int'].tolist()
    target_id     = int(user_df.loc[target_idx, 'item_id_int'])
 
    return X_shots, shot_item_ids, target_id
 
 
def evaluate_fewshot(global_model, unseen_users, df, emb_map,
                     all_metas_gpu, num_shots,
                     k=10, device='cuda', finetune_epochs=5, lr=0.01):
    """
    Valuta gli unseen users con few-shot adaptation.
    num_shots: int (1, 2, 3, ...) oppure None per "full"
    """
    label = f"{num_shots}-shot" if num_shots is not None else "full"
 
    num_items    = all_metas_gpu.shape[0]
    global_state = global_model.state_dict()
    all_ids_set  = set(range(num_items))
    all_ids_arr  = np.arange(num_items)
    use_amp      = (device == 'cuda')
 
    # [OPT-7] Pre-computa embedding item una volta sola
    global_model.eval()
    with torch.no_grad():
        chunk = 4096
        all_item_embs = torch.cat([
            global_model.get_item_repr(all_metas_gpu[i:i + chunk])
            for i in range(0, num_items, chunk)
        ], dim=0)
 
    hits, ndcgs, count = 0, 0, 0
 
    for user_id in tqdm(unseen_users, desc=f"Few-shot eval ({label})", leave=False):
        X_shots, shot_item_ids, target_id = get_fewshot_data(
            user_id, df, emb_map, num_shots
        )
        if X_shots is None:
            continue
 
        # Parte sempre dai pesi globali (nessuna MLP pre-allenata per unseen users)
        local_model = TwoTowerRecommender().to(device)
        local_model.load_state_dict(global_state, strict=False)
 
        X_shots_dev   = X_shots.to(device)
        shot_ids_set  = set(shot_item_ids)
        eligible_neg  = all_ids_arr[~np.isin(all_ids_arr, list(shot_ids_set))]
 
        # Fine-tuning della sola client_mlp sulle K interazioni disponibili
        local_model.train()
        optimizer = torch.optim.Adam(local_model.client_mlp.parameters(), lr=lr)
 
        if len(shot_item_ids) > 0 and len(eligible_neg) > 0:
            batch_pos = shot_item_ids if len(shot_item_ids) < 32 \
                        else random.sample(shot_item_ids, 32)
            pos_t = torch.tensor(batch_pos, device=device)
 
            with torch.amp.autocast(device_type=device, enabled=use_amp):
                for _ in range(finetune_epochs):
                    optimizer.zero_grad()
                    user_repr = local_model.get_user_repr(X_shots_dev)
                    pos_reprs = local_model.get_item_repr(all_metas_gpu[pos_t])
                    n_neg     = min(len(batch_pos), len(eligible_neg))
                    neg_idx   = np.random.choice(eligible_neg, size=n_neg, replace=False)
                    neg_t     = torch.tensor(neg_idx, device=device)
                    neg_reprs = local_model.get_item_repr(all_metas_gpu[neg_t])
                    loss = bpr_loss(
                        local_model.training_score(user_repr, pos_reprs),
                        local_model.training_score(user_repr, neg_reprs)
                    )
                    loss.backward()
                    optimizer.step()
 
        # Inferenza
        local_model.eval()
        with torch.no_grad():
            user_repr  = local_model.get_user_repr(X_shots_dev)
            neg_cands  = list(all_ids_set - shot_ids_set - {target_id})
            neg_arr    = np.array(neg_cands)
            neg_embs   = all_item_embs[neg_arr]
            target_emb = all_item_embs[target_id].unsqueeze(0)
 
            with torch.amp.autocast(device_type=device, enabled=use_amp):
                target_score = local_model.score(user_repr, target_emb).item()
                neg_scores   = local_model.score(user_repr, neg_embs)
 
            all_scores = torch.cat([torch.tensor([target_score], device=device), neg_scores])
            top_k_idx  = torch.topk(all_scores, k).indices.cpu().numpy()
 
            if 0 in top_k_idx:
                hits += 1
                rank = int(np.where(top_k_idx == 0)[0][0])
                ndcgs += 1.0 / np.log2(rank + 2)
            count += 1
 
    if count == 0:
        return 0.0, 0.0
    print(f"  [{label}] utenti valutati: {count}/{len(unseen_users)}")
    return hits / count, ndcgs / count

## Split utenti

In [49]:
# =============================================================================
# SPLIT UTENTI
# =============================================================================

def split_users(df, unseen_ratio=0.2, seed=42):
    rng = np.random.RandomState(seed)
    all_users = df['user_id_int'].unique()
    rng.shuffle(all_users)
    n_total = len(all_users)
    n_unseen  = int(n_total * unseen_ratio)
    unseen_users  = all_users[:n_unseen]
    train_users = all_users[n_unseen:]
    return train_users, unseen_users

## Definizione esperimenti

In [50]:
def run_experiment(seed):
    print(f"\n===== RUN con seed {seed} =====")
    set_seed(seed)

    train_users, unseen_users = split_users(
        df_sampled, unseen_ratio=0.2, seed=seed
    )
    print(f"Train users: {len(train_users)}")
    print(f"Unseen users:   {len(unseen_users)}")

    # -- CONFIGURAZIONE --
    LR                    = 0.0005
    LOCAL_EPOCHS          = 3
    NUM_NEG_TRAIN         = 10
    USE_HARD_NEG          = True
    CLIENTS_PER_ROUND     = round(0.05 * len(train_users)) # 0.05
    GLOBAL_ROUNDS         = 100
    EVAL_EVERY            = 5
    INFERENCE_TEMPERATURE = 0.07
    FEDAVG_MOMENTUM       = 0.9
    K                     = 20
    EVAL_FRACTION         = 1
    LR_WARMUP_STEPS   = 10

    client_states   = {user_id: None for user_id in train_users}
    best_val_hr    = 0.0
    best_val_ndcg  = 0.0
    best_state      = None
    best_client_states = None
    momentum_buffer = None

    global_model = TwoTowerRecommender(
        inference_temperature=INFERENCE_TEMPERATURE
    ).to(device)

    print(f"\n=== Inizio Training Federato con seed = {seed} ===")
    print(f"{'Round':<6} | {'Loss':<8} | {'HR@' + str(K):<8} | {'NDCG@' + str(K):<8}")
    print("-" * 45)

    for round_num in range(1, GLOBAL_ROUNDS + 1):
        local_weights = []
        local_sizes   = []
        local_losses  = []

        # [OPT-2] state_dict() del modello globale calcolato UNA VOLTA per round
        # e condiviso tra tutti i client del round (in lettura).
        # Il codice originale faceva copy.deepcopy() dentro il loop per ogni client,
        # che significa N_clients deep copy per round — inutile perché nessun
        # client modifica il dict condiviso (ogni client crea il suo local_model).
        round_state_dict = global_model.state_dict()

        selected = np.random.choice(train_users, CLIENTS_PER_ROUND, replace=False)
        for user_id in selected:
            train_data, _ = get_client_data(user_id, df_sampled, review_emb_map)
            if train_data is None:
                continue
            X_train, train_item_ids = train_data

            w, loss, n = train_client(
                user_id,
                round_state_dict,       # [OPT-2] riferimento condiviso, no deepcopy
                X_train,
                train_item_ids,
                item_meta_tensor_gpu,   # [OPT-1] già su GPU
                device,
                client_states,
                lr=LR,
                epochs=LOCAL_EPOCHS,
                num_neg=NUM_NEG_TRAIN,
                use_hard_negatives=USE_HARD_NEG,
                lr_warmup_steps=LR_WARMUP_STEPS,
                current_step=round_num - 1,
                total_steps=GLOBAL_ROUNDS
            )

            local_weights.append(w)
            local_sizes.append(n)
            local_losses.append(loss)

        if not local_weights:
            continue

        momentum_buffer = weighted_fedavg_momentum(
            global_model, local_weights, local_sizes,
            momentum_buffer, beta=FEDAVG_MOMENTUM
        )

        avg_loss = sum(local_losses) / len(local_losses)

        if round_num % EVAL_EVERY == 0:
            val_hr, val_ndcg = evaluate_top_k(
                global_model, train_users, df_sampled,
                review_emb_map, item_meta_tensor_gpu,   # [OPT-1]
                client_states, k=K, device=device, mode="val",
                eval_fraction=EVAL_FRACTION
            )

            marker = ""
            if val_hr > best_val_hr:
                best_val_hr        = val_hr
                best_state         = copy.deepcopy(global_model.state_dict())
                best_client_states = copy.deepcopy(client_states)
                marker = "  <- Best"
 
            print(f"{round_num:<6} | {avg_loss:<8.4f} | {val_hr:<10.4f} | {val_ndcg:<10.4f} {marker}")
        else:
            print(f"{round_num:<6} | {avg_loss:<8.4f} |")

    print("\n=== Fine Training ===")

    # =========================================================================
    # TEST FINALE sul best model
    # =========================================================================
    global_model.load_state_dict(best_state)
     
    # 1. Test warm users (ultima interazione, MLP già allenata)
    print("\n--- TEST WARM USERS (ultima interazione) ---")
    warm_hr, warm_ndcg = evaluate_top_k(
        global_model, train_users, df_sampled,
        review_emb_map, item_meta_tensor_gpu,
        best_client_states, k=K, device=device, mode="test"
    )
    print(f"Warm  HR@{K}: {warm_hr:.4f}  |  NDCG@{K}: {warm_ndcg:.4f}")

    # 2. Few-shot test sugli unseen users: 1-shot, 2-shot, 3-shot, full
    print("\n--- TEST UNSEEN USERS (few-shot adaptation) ---")
    shot_configs = [1, 2, 3, None]   # None = full
    fewshot_results = {}
 
    for num_shots in shot_configs:
        label = f"{num_shots}-shot" if num_shots is not None else "full"
        hr, ndcg = evaluate_fewshot(
            global_model, unseen_users, df_sampled,
            review_emb_map, item_meta_tensor_gpu,
            num_shots=num_shots, k=K, device=device,
            finetune_epochs=5, lr=0.005
        )
        fewshot_results[label] = (hr, ndcg)
        print(f"  {label:<8}  HR@{K}: {hr:.4f}  |  NDCG@{K}: {ndcg:.4f}")
 
    return warm_hr, warm_ndcg, fewshot_results

In [51]:
# =============================================================================
# MAIN — 5 RUN CON SEED DIVERSI
# =============================================================================
 
seeds      = [0] # 0, 1, 2, 3, 4
shot_labels = ["1-shot", "2-shot", "3-shot", "full"]
 
# Accumula risultati per seed
warm_hrs, warm_ndcgs = [], []
fewshot_hrs  = {l: [] for l in shot_labels}
fewshot_ndcgs = {l: [] for l in shot_labels}
 
for s in seeds:
    warm_hr, warm_ndcg, fewshot_results = run_experiment(s)
    warm_hrs.append(warm_hr)
    warm_ndcgs.append(warm_ndcg)
    for label in shot_labels:
        fewshot_hrs[label].append(fewshot_results[label][0])
        fewshot_ndcgs[label].append(fewshot_results[label][1])
 
K = 20
 
print("\n" + "=" * 50)
print("RISULTATI FINALI (media ± std su 5 seed)")
print("=" * 50)
 
print(f"\n{'Scenario':<12} | {'HR@'+str(K):<18} | {'NDCG@'+str(K):<18}")
print("-" * 55)
 
# Warm users
m_hr   = np.mean(warm_hrs);   s_hr   = np.std(warm_hrs)
m_ndcg = np.mean(warm_ndcgs); s_ndcg = np.std(warm_ndcgs)
print(f"{'warm':<12} | {m_hr:.4f} ± {s_hr:.4f}   | {m_ndcg:.4f} ± {s_ndcg:.4f}")
 
# Few-shot unseen users
for label in shot_labels:
    m_hr   = np.mean(fewshot_hrs[label]);   s_hr   = np.std(fewshot_hrs[label])
    m_ndcg = np.mean(fewshot_ndcgs[label]); s_ndcg = np.std(fewshot_ndcgs[label])
    print(f"{label:<12} | {m_hr:.4f} ± {s_hr:.4f}   | {m_ndcg:.4f} ± {s_ndcg:.4f}")


===== RUN con seed 0 =====
Train users: 3501
Unseen users:   875

=== Inizio Training Federato con seed = 0 ===
Round  | Loss     | HR@20    | NDCG@20 
---------------------------------------------
1      | 0.7089   |
2      | 0.6866   |
3      | 0.6696   |
4      | 0.6549   |


Evaluating (val): 100%|██████████| 3501/3501 [01:50<00:00, 31.82it/s]


5      | 0.6427   | 0.0580     | 0.0329       <- Best
6      | 0.6321   |
7      | 0.6188   |
8      | 0.6086   |
9      | 0.5986   |


Evaluating (val): 100%|██████████| 3501/3501 [01:51<00:00, 31.43it/s]


10     | 0.5778   | 0.0588     | 0.0325       <- Best
11     | 0.5785   |
12     | 0.5766   |
13     | 0.5618   |
14     | 0.5678   |


...

Warm  HR@20: 0.0411  |  NDCG@20: 0.0208

--- TEST UNSEEN USERS (few-shot adaptation) ---


  [1-shot] utenti valutati: 875/875
  1-shot    HR@20: 0.1017  |  NDCG@20: 0.0638


  [2-shot] utenti valutati: 875/875
  2-shot    HR@20: 0.1017  |  NDCG@20: 0.0611


  [3-shot] utenti valutati: 875/875
  3-shot    HR@20: 0.0994  |  NDCG@20: 0.0591


  [full] utenti valutati: 875/875
  full      HR@20: 0.0503  |  NDCG@20: 0.0257

RISULTATI FINALI (media ± std su 5 seed)

Scenario     | HR@20              | NDCG@20           
-------------------------------------------------------
warm         | 0.0411 ± 0.0000   | 0.0208 ± 0.0000
1-shot       | 0.1017 ± 0.0000   | 0.0638 ± 0.0000
2-shot       | 0.1017 ± 0.0000   | 0.0611 ± 0.0000
3-shot       | 0.0994 ± 0.0000   | 0.0591 ± 0.0000
full         | 0.0503 ± 0.0000   | 0.0257 ± 0.0000


In [ ]:
#import numpy as np

def analyze_target_popularity(df, train_users, unseen_users, num_shots_list):
    """
    Calcola la popolarità media (numero di interazioni nel dataset) 
    degli item target per warm e few-shot users.
    """
    # Conta la popolarità di ogni item una volta sola
    item_popularity = df['item_id_int'].value_counts().to_dict()
    
    stats = {}
    
    # 1. Analisi Warm Users (Ultima interazione)
    warm_pops = []
    for uid in train_users:
        user_df = df[df['user_id_int'] == uid].sort_values('timestamp')
        if len(user_df) < 3: continue
        target_item = user_df.iloc[-1]['item_id_int'] # Ultima interazione
        warm_pops.append(item_popularity.get(target_item, 0))
    stats['Warm'] = {'mean': np.mean(warm_pops), 'median': np.median(warm_pops)}
    
    # 2. Analisi Few-Shot Users (Interazione num_shots)
    for n_shots in num_shots_list:
        fewshot_pops = []
        for uid in unseen_users:
            user_df = df[df['user_id_int'] == uid].sort_values('timestamp')
            if len(user_df) < n_shots + 1: continue
            
            # Target è l'interazione alla posizione num_shots
            # (indice n_shots perché 0-based)
            target_item = user_df.iloc[n_shots]['item_id_int'] 
            fewshot_pops.append(item_popularity.get(target_item, 0))
            
        stats[f'{n_shots}-shot'] = {'mean': np.mean(fewshot_pops), 'median': np.median(fewshot_pops)}
        
    return stats

train_users, unseen_users = split_users(
    df_sampled, unseen_ratio=0.2, seed=0
)

# Esegui l'analisi
popularity_stats = analyze_target_popularity(
    df_sampled, train_users, unseen_users, num_shots_list=[1, 2, 3]
)

print("Popolarità Media dei Target Item (conteggio interazioni nel dataset):")
for scenario, vals in popularity_stats.items():
    print(f"{scenario:<10} | Media: {vals['mean']:.1f} | Mediana: {vals['median']:.1f}")

In [ ]:
def analyze_target_similarity(df, train_users, unseen_users, item_meta_tensor_gpu, num_shots_list):
    """
    Calcola la similarità coseno media tra la rappresentazione del training set
    e l'item target per warm e few-shot users.
    """
    device = item_meta_tensor_gpu.device
    stats = {}

    # 1. Analisi Warm Users
    warm_sims = []
    for uid in train_users:
        user_df = df[df['user_id_int'] == uid].sort_values('timestamp')
        if len(user_df) < 3: continue
        
        indices = user_df.index.tolist()
        train_idx = indices[:-2]
        target_idx = indices[-1]
        
        # Media degli embedding degli item di train
        train_items = user_df.loc[train_idx, 'item_id_int'].tolist()
        train_embs = item_meta_tensor_gpu[torch.tensor(train_items, device=device)]
        mean_train_emb = train_embs.mean(dim=0)
        
        # Embedding del target
        target_item = user_df.loc[target_idx, 'item_id_int']
        target_emb = item_meta_tensor_gpu[target_item]
        
        # Similarità coseno
        sim = torch.nn.functional.cosine_similarity(mean_train_emb.unsqueeze(0), target_emb.unsqueeze(0)).item()
        warm_sims.append(sim)
        
    stats['Warm'] = {'mean_sim': np.mean(warm_sims)}

    # 2. Analisi Few-Shot Users
    for n_shots in num_shots_list:
        fewshot_sims = []
        for uid in unseen_users:
            user_df = df[df['user_id_int'] == uid].sort_values('timestamp')
            if len(user_df) < n_shots + 1: continue
            
            indices = user_df.index.tolist()
            shot_idx = indices[:n_shots]
            target_idx = indices[n_shots]
            
            # Media degli embedding degli item di shot
            shot_items = user_df.loc[shot_idx, 'item_id_int'].tolist()
            shot_embs = item_meta_tensor_gpu[torch.tensor(shot_items, device=device)]
            mean_shot_emb = shot_embs.mean(dim=0)
            
            # Embedding del target
            target_item = user_df.loc[target_idx, 'item_id_int']
            target_emb = item_meta_tensor_gpu[target_item]
            
            sim = torch.nn.functional.cosine_similarity(mean_shot_emb.unsqueeze(0), target_emb.unsqueeze(0)).item()
            fewshot_sims.append(sim)
            
        stats[f'{n_shots}-shot'] = {'mean_sim': np.mean(fewshot_sims)}
        
    return stats

# Esegui l'analisi (richiede item_meta_tensor_gpu caricato in memoria come nel tuo codice)
sim_stats = analyze_target_similarity(df_sampled, train_users, unseen_users, item_meta_tensor_gpu, num_shots_list=[1, 2, 3])

print("Similarità Coseno Media (Train vs Target):")
for scenario, vals in sim_stats.items():
    print(f"{scenario:<10} | Sim Mean: {vals['mean_sim']:.4f}")

In [ ]:
def analyze_full_protocol(df, train_users, unseen_users, 
                          item_meta_tensor_gpu, num_shots_list):
    device = item_meta_tensor_gpu.device
    stats = {}

    # Warm: contesto = tutto tranne ultime 2, target = ultima
    warm_sims, warm_gaps = [], []
    for uid in train_users:
        user_df = df[df['user_id_int'] == uid].sort_values('timestamp')
        if len(user_df) < 3: continue
        indices = user_df.index.tolist()
        train_items = user_df.loc[indices[:-2], 'item_id_int'].tolist()
        target_item = int(user_df.loc[indices[-1], 'item_id_int'])
        
        train_embs = item_meta_tensor_gpu[torch.tensor(train_items, device=device)]
        target_emb = item_meta_tensor_gpu[target_item]
        sim = F.cosine_similarity(train_embs.mean(0, keepdim=True), 
                                   target_emb.unsqueeze(0)).item()
        warm_sims.append(sim)
        # Gap temporale: distanza in posizione tra ultimo train e target
        warm_gaps.append(2)  # sempre 2 posizioni di distanza (salta penultima)
    
    stats['warm'] = {
        'cos_sim': np.mean(warm_sims),
        'temporal_gap': np.mean(warm_gaps),
        'context_size': np.mean([
            len(df[df['user_id_int']==uid]) - 2 
            for uid in train_users 
            if len(df[df['user_id_int']==uid]) >= 3
        ])
    }

    # Few-shot: contesto = prime K, target = K+1-esima
    for n in num_shots_list:
        sims, gaps, ctx_sizes = [], [], []
        for uid in unseen_users:
            user_df = df[df['user_id_int'] == uid].sort_values('timestamp')
            if len(user_df) < n + 1: continue
            indices = user_df.index.tolist()
            shot_items = user_df.loc[indices[:n], 'item_id_int'].tolist()
            target_item = int(user_df.loc[indices[n], 'item_id_int'])
            
            shot_embs = item_meta_tensor_gpu[torch.tensor(shot_items, device=device)]
            target_emb = item_meta_tensor_gpu[target_item]
            sim = F.cosine_similarity(shot_embs.mean(0, keepdim=True),
                                       target_emb.unsqueeze(0)).item()
            sims.append(sim)
            gaps.append(1)   # sempre 1 posizione di distanza
            ctx_sizes.append(n)
        
        stats[f'{n}-shot'] = {
            'cos_sim': np.mean(sims),
            'temporal_gap': np.mean(gaps),
            'context_size': n
        }

    return stats

stats = analyze_full_protocol(df_sampled, train_users, unseen_users, 
                               item_meta_tensor_gpu, [1, 2, 3])
print(f"{'Scenario':<10} | {'Cos Sim':>8} | {'Temp Gap':>9} | {'Ctx Size':>9}")
print("-" * 45)
for k, v in stats.items():
    print(f"{k:<10} | {v['cos_sim']:>8.4f} | {v['temporal_gap']:>9.1f} | {v['context_size']:>9.1f}")

In [ ]:
def analyze_temporal_correlation(df, train_users, unseen_users, 
                                  item_meta_tensor_gpu, num_shots_list):
    """
    Misura la similarità coseno tra l'item immediatamente precedente 
    al target e il target stesso — proxy della correlazione temporale locale.
    """
    device = item_meta_tensor_gpu.device
    stats = {}

    # Warm: l'item immediatamente precedente al target è la penultima
    # interazione, che però NON è nel contesto (get_client_data usa indices[:-2])
    # Quindi la "distanza" reale è tra l'ultimo item del contesto e il target,
    # con la penultima in mezzo — gap effettivo = 2
    warm_local_sims = []
    for uid in train_users:
        user_df = df[df['user_id_int'] == uid].sort_values('timestamp')
        if len(user_df) < 3: continue
        items = user_df['item_id_int'].tolist()
        # Ultimo item del contesto = items[-3], target = items[-1]
        last_ctx_emb = item_meta_tensor_gpu[items[-3]]
        target_emb   = item_meta_tensor_gpu[items[-1]]
        sim = F.cosine_similarity(last_ctx_emb.unsqueeze(0), 
                                   target_emb.unsqueeze(0)).item()
        warm_local_sims.append(sim)
    stats['warm'] = {'local_sim': np.mean(warm_local_sims)}

    # Few-shot: l'item immediatamente precedente al target è l'ultimo shot
    # gap effettivo = 1
    for n in num_shots_list:
        local_sims = []
        for uid in unseen_users:
            user_df = df[df['user_id_int'] == uid].sort_values('timestamp')
            if len(user_df) < n + 1: continue
            items = user_df['item_id_int'].tolist()
            # Ultimo shot = items[n-1], target = items[n]
            last_shot_emb = item_meta_tensor_gpu[items[n-1]]
            target_emb    = item_meta_tensor_gpu[items[n]]
            sim = F.cosine_similarity(last_shot_emb.unsqueeze(0),
                                       target_emb.unsqueeze(0)).item()
            local_sims.append(sim)
        stats[f'{n}-shot'] = {'local_sim': np.mean(local_sims)}

    return stats

local_stats = analyze_temporal_correlation(
    df_sampled, train_users, unseen_users, 
    item_meta_tensor_gpu, [1, 2, 3]
)
print(f"{'Scenario':<10} | {'Local Sim (last→target)':>22}")
print("-" * 36)
for k, v in local_stats.items():
    print(f"{k:<10} | {v['local_sim']:>22.4f}")

In [ ]:

train_users, unseen_users = split_users(
    df_sampled, unseen_ratio=0.2, seed=0
)
print(f"Train users: {len(train_users)}")
print(f"Unseen users:   {len(unseen_users)}")

# Distribuzione interazioni per i due gruppi
train_counts  = df_sampled[df_sampled['user_id_int'].isin(train_users)]\
                .groupby('user_id_int').size()
unseen_counts = df_sampled[df_sampled['user_id_int'].isin(unseen_users)]\
                .groupby('user_id_int').size()

print(f"Train  — media: {train_counts.mean():.1f}, mediana: {train_counts.median():.0f}, min: {train_counts.min()}")
print(f"Unseen — media: {unseen_counts.mean():.1f}, mediana: {unseen_counts.median():.0f}, min: {unseen_counts.min()}")

In [ ]:
# =============================================================================
# CURVA FEW-SHOT — HR@K e NDCG@K al variare degli shot
# =============================================================================
import matplotlib.pyplot as plt
 
x_labels = shot_labels  # ["1-shot", "2-shot", "3-shot", "full"]
x_pos    = np.arange(len(x_labels))
 
hr_means   = [np.mean(fewshot_hrs[l])   for l in shot_labels]
hr_stds    = [np.std(fewshot_hrs[l])    for l in shot_labels]
ndcg_means = [np.mean(fewshot_ndcgs[l]) for l in shot_labels]
ndcg_stds  = [np.std(fewshot_ndcgs[l]) for l in shot_labels]
 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
 
# HR@K
ax1.errorbar(x_pos, hr_means, yerr=hr_stds, marker='o', capsize=4,
             linewidth=2, color='steelblue', label='ReLoG (unseen users)')
ax1.axhline(np.mean(warm_hrs), linestyle='--', color='darkorange',
            linewidth=1.5, label='Warm users (reference)')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels)
ax1.set_ylabel(f"HR@{K}")
ax1.set_title(f"HR@{K} — Few-shot adaptation")
ax1.legend()
ax1.grid(True, alpha=0.3)
 
# NDCG@K
ax2.errorbar(x_pos, ndcg_means, yerr=ndcg_stds, marker='s', capsize=4,
             linewidth=2, color='seagreen', label='ReLoG (unseen users)')
ax2.axhline(np.mean(warm_ndcgs), linestyle='--', color='darkorange',
            linewidth=1.5, label='Warm users (reference)')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(x_labels)
ax2.set_ylabel(f"NDCG@{K}")
ax2.set_title(f"NDCG@{K} — Few-shot adaptation")
ax2.legend()
ax2.grid(True, alpha=0.3)
 
plt.suptitle("ReLoG — Few-shot user adaptation (unseen users)", fontsize=13)
plt.tight_layout()
plt.savefig("fewshot_curve.pdf", bbox_inches='tight')  # per il paper
plt.savefig("fewshot_curve.png", dpi=150, bbox_inches='tight')
plt.show()
print("Curva salvata in fewshot_curve.pdf e fewshot_curve.png")